# Executor black-box launcher (`libero_object` task 0)

Runs a **single injected arm skill** through the real executor + real VLA server + real LIBERO env.

Scope:
- plan goal comes from `env.task_description`
- plan contains exactly one arm skill (`arm.transfer`)
- pass/fail gate is `env.success()`
- test entrypoint: `tests/test_executor_live_libero.py`

## 1 - Clone repo

In [ ]:
import getpass

github_token = getpass.getpass('GitHub PAT (repo scope): ')

In [ ]:
import os

if os.path.exists('/content/Longor'):
    os.chdir('/content/Longor')
    print('pulling repo...')
    !git pull
else:
    os.environ['CLONE_URL'] = f"https://{github_token}@github.com/pmish11/Longor-Robotics.git"
    !git clone $CLONE_URL "/content/Longor"
    os.chdir('/content/Longor')

## 2 - Headless rendering + app deps

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

!apt-get update
!apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r deps/requirements-libero.txt
%pip install -e .

# Silence robosuite macro recommendation warning for this runtime.
!python -m robosuite.scripts.setup_macros

## 3 - Pi0.5 (OpenPI) venv

In [ ]:
!test -d /content/Longor/.vla-venv || python3 -m venv /content/Longor/.vla-venv
!/content/Longor/.vla-venv/bin/pip install -U pip
!test -d /content/openpi || git clone --recurse-submodules https://github.com/Physical-Intelligence/openpi /content/openpi
# pip ignores OpenPI's [tool.uv.sources] lerobot pin. Pass the same git rev as
# deps/requirements-vla-pi05-linux.txt in this command or pip backtracks on
# PyPI lerobot (numpy>=2 vs OpenPI numpy<2).
!/content/Longor/.vla-venv/bin/pip install -e /content/openpi "lerobot @ git+https://github.com/huggingface/lerobot@0cf864870cf29f4738d3ade893e6fd13fbd7cdb5"
!/content/Longor/.vla-venv/bin/pip install -r deps/requirements-vla-pi05-linux.txt
!/content/Longor/.vla-venv/bin/pip install -e .

## 4 - Start VLA server + health

In [ ]:
import os, subprocess, time, urllib.request

with open('.env', 'w', encoding='utf-8') as f:
    f.write('LOG_LEVEL=DEBUG\nSAVE_VIDEO=0\n')

VLA_LOG = '/tmp/executor_blackbox_vla_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'
openpi_root = '/content/openpi'
env = dict(os.environ)

# OpenPI sometimes isn't importable even after editable install; make its source visible.
extra_paths = [
    openpi_root,
    f'{openpi_root}/src',
    f'{openpi_root}/packages/openpi/src',
]
py_path = env.get('PYTHONPATH', '')
py_parts = [p for p in py_path.split(':') if p]
for path in extra_paths:
    if os.path.isdir(path) and path not in py_parts:
        py_parts.append(path)
env['PYTHONPATH'] = ':'.join(py_parts)

# Colab gsutil can fail on OpenPI composite objects when crcmod C extension is unavailable.
# For this notebook run, skip hash checks so checkpoint download can complete.
boto_cfg = '/tmp/executor_blackbox.boto'
with open(boto_cfg, 'w', encoding='utf-8') as f:
    f.write('[GSUtil]\ncheck_hashes = never\n')
env['BOTO_CONFIG'] = boto_cfg

check = subprocess.run(
    [vla_python, '-c', 'import openpi'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
if check.returncode != 0:
    print('OpenPI import failed in VLA venv; retrying editable install...')
    reinstall = subprocess.run(
        [vla_python, '-m', 'pip', 'install', '-e', openpi_root],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if reinstall.returncode != 0:
        print(reinstall.stdout)
        raise RuntimeError('Failed reinstalling OpenPI editable package in .vla-venv')

    check = subprocess.run(
        [vla_python, '-c', 'import openpi'],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if check.returncode != 0:
        print(check.stdout)
        raise RuntimeError('OpenPI still not importable in .vla-venv')

log = open(VLA_LOG, 'w', buffering=1)
proc = subprocess.Popen(
    [vla_python, 'services/run_vla_server.py'],
    cwd='/content/Longor',
    stdout=log,
    stderr=subprocess.STDOUT,
    env=env,
)

health_ok = False
max_wait_s = 1800  # First OpenPI checkpoint download is large; allow enough time.
for elapsed in range(0, max_wait_s, 5):
    if proc.poll() is not None:
        raise RuntimeError(f'VLA server exited early with code {proc.returncode}; see {VLA_LOG}')
    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        print('Health:', resp.read().decode())
        health_ok = True
        break
    except Exception:
        print(f'Waiting for /health ({elapsed}s)...')
        time.sleep(5)

if not health_ok:
    raise RuntimeError(f'VLA server not healthy after {max_wait_s}s; see {VLA_LOG}')

In [ ]:
import re

BENIGN_PATTERNS = [
    r"Found no hashes to validate object downloaded",
    r"Could not determine bucket type for bucket name big_vision",
]

with open('/tmp/executor_blackbox_vla_server.log', 'r', encoding='utf-8', errors='replace') as f:
    lines = [line.rstrip('\n') for line in f]

# Print startup highlights.
for line in lines:
    if (
        'Loading Pi0.5' in line
        or 'Policy loaded' in line
        or '/health' in line
        or 'Application startup complete' in line
    ):
        print(line)

# Print non-benign warnings/errors only.
for line in lines:
    is_warn_or_err = ('WARNING' in line) or ('ERROR' in line) or ('Traceback' in line)
    is_benign = any(re.search(pattern, line) for pattern in BENIGN_PATTERNS)
    if is_warn_or_err and not is_benign:
        print(line)

## 5 - Run executor black-box integration test

In [ ]:
# Keep signal high: ignore known third-party deprecation noise only.
!pytest -m integration tests/test_executor_live_libero.py -q -s -W "ignore:The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`.:UserWarning"